In [1]:
import os
import pandas as pd, numpy as np
import ipaddress

In [2]:
source_folder = "../data/cicids2017/hhuang_fix/" # this is the folder that contains the four CSV files obtained after downloading and extracting the dataset
output_folder = os.path.join(source_folder, "pragmatic_assessment") # the output of this notebook will be saved in this folder

file_names = [
    "monday.csv",
    "tuesday.csv",
    "wednesday.csv",
    "thursday.csv",
    "friday.csv"
]

df = pd.DataFrame()

for f in file_names:
    input_file = source_folder + f
    temp_df = pd.read_csv(input_file)
    print("Read {} lines...".format(len(temp_df)))
    #df = df.append(temp_df) # We used this command since we used pandas < 2.0
    df = pd.concat([df, temp_df], ignore_index=True) # Use this if pandas >= 2.0
    print("\t...total length: {}".format(len(df)))

print("...done!")

Read 372341 lines...
	...total length: 372341
Read 322538 lines...
	...total length: 694879
Read 497042 lines...
	...total length: 1191921
Read 363792 lines...
	...total length: 1555713
Read 548602 lines...
	...total length: 2104315
...done!


In [3]:
df['Label'].unique()

<ArrowStringArray>
[                                'BENIGN',
                'FTP-Patator - Attempted',
                            'FTP-Patator',
                            'SSH-Patator',
                          'DoS Slowloris',
                       'DoS Slowhttptest',
           'DoS Slowhttptest - Attempted',
              'DoS Slowloris - Attempted',
                               'DoS Hulk',
                   'DoS Hulk - Attempted',
                          'DoS GoldenEye',
                             'Heartbleed',
   'Web Attack - Brute Force - Attempted',
               'Web Attack - Brute Force',
                           'Infiltration',
                'Infiltration - Portscan',
           'Web Attack - XSS - Attempted',
                       'Web Attack - XSS',
 'Web Attack - SQL Injection - Attempted',
               'Infiltration - Attempted',
             'Web Attack - SQL Injection',
                     'Botnet - Attempted',
                                 'B

In [4]:
def fixValues(df):
    # Function to fix NaNs and Infinite values
    # NaNs are replaced with the MEAN
    # Infinite are replaced with the MAX
    
    
    import numpy as np
    x = df#.copy(deep=True)
    for c in x.columns:
        if x[c].dtype == 'int' or x[c].dtype == 'float':
            temp = np.asarray(x[c], dtype=np.float64)
            # remove NaN & Infinity (if there are)
            temp = temp[np.isfinite(temp)]
            mean_value = temp.mean()
            max_value = temp.max()
            x[c].replace([np.inf, -np.inf], max_value, inplace=True)
            x[c].replace([np.nan], mean_value, inplace=True)
    
    return x


def uniformPorts(df, srcPort_name, dstPort_name):
    # Function to uniformize well-known, registered and dynamic ports 

    df.rename({srcPort_name: 'SrcPort', dstPort_name: 'DstPort'}, axis=1, inplace=True)
        
    #converting strings to numeric
    df['SrcPort_num'] = pd.to_numeric(df['SrcPort'], errors='coerce')
    df['SrcPort_num'] = df['SrcPort_num'].replace([np.nan], -1)
    df['DstPort_num'] = pd.to_numeric(df['DstPort'], errors='coerce')
    df['DstPort_num'] = df['DstPort_num'].replace([np.nan], -1)
    #determining low&high ports
    srcPort_conditions = [
        (df['SrcPort_num'] == -1),
        (df['SrcPort_num'] >= 0) & (df['SrcPort_num'] <= 1023),
        (df['SrcPort_num'] >= 1024) & (df['SrcPort_num'] <= 49151),
        (df['SrcPort_num'] > 49151)
    ]
    dstPort_conditions = [
        (df['DstPort_num'] == -1),
        (df['DstPort_num'] >= 0) & (df['DstPort_num'] <= 1023),
        (df['DstPort_num'] >= 1024) & (df['DstPort_num'] <= 49151),
        (df['DstPort_num'] > 49151)
    ]    
    port_choices = ['none','well-known','registered','dynamic']
    df['SrcPort_type'] = np.select(srcPort_conditions, port_choices, default='unknown')
    df['DstPort_type'] = np.select(dstPort_conditions, port_choices, default='unknown')
    
    return df

def uniformIP(df, srcIP_name, dstIP_name, internal_network, *, internal_network2 = None):
    # Function for assigning IPs to internal/external network
    df.rename({srcIP_name: 'SrcIP', dstIP_name: 'DstIP'}, axis=1, inplace=True)
    
    if internal_network2 == None:
        df['SrcIP_internal'] = df['SrcIP'].apply(ipaddress.ip_address).isin(ipaddress.ip_network(internal_network))
        df['DstIP_internal'] = df['DstIP'].apply(ipaddress.ip_address).isin(ipaddress.ip_network(internal_network))
    else:
        df['SrcIP_internal1'] = df['SrcIP'].apply(ipaddress.ip_address).isin(ipaddress.ip_network(internal_network))
        df['DstIP_internal1'] = df['DstIP'].apply(ipaddress.ip_address).isin(ipaddress.ip_network(internal_network))
        df['SrcIP_internal2'] = df['SrcIP'].apply(ipaddress.ip_address).isin(ipaddress.ip_network(internal_network2))
        df['DstIP_internal2'] = df['DstIP'].apply(ipaddress.ip_address).isin(ipaddress.ip_network(internal_network2))
        
        df['DstIP_internal'] = (df['DstIP_internal1']) | (df['DstIP_internal2'])
        df['SrcIP_internal'] = (df['SrcIP_internal1']) | (df['SrcIP_internal2'])
        
        df.drop(columns=['SrcIP_internal1', 'SrcIP_internal2', 'DstIP_internal1', 'DstIP_internal2'], inplace=True)
        
    # check internal/external
    int_int = df.loc[(df['SrcIP_internal'] == True) & (df['DstIP_internal'] == True)]
    int_ext = df.loc[(df['SrcIP_internal'] == True) & (df['DstIP_internal'] == False)]
    ext_int = df.loc[(df['SrcIP_internal'] == False) & (df['DstIP_internal'] == True)]
    ext_ext = df.loc[(df['SrcIP_internal'] == False) & (df['DstIP_internal'] == False)]

    print("int_int = {}\n int_ext = {}\n ext_int = {}\n ext_ext = {}".format(len(int_int), len(int_ext), len(ext_int), len(ext_ext)))
        
    return df

In [5]:
df.columns = df.columns.str.replace(' ', '')
df = fixValues(df)
df['Label'].unique()

/tmp/ipykernel_69162/3437851558.py:16: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  x[c].replace([np.inf, -np.inf], max_value, inplace=True)
/tmp/ipykernel_69162/3437851558.py:17: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method ne

<ArrowStringArray>
[                                'BENIGN',
                'FTP-Patator - Attempted',
                            'FTP-Patator',
                            'SSH-Patator',
                          'DoS Slowloris',
                       'DoS Slowhttptest',
           'DoS Slowhttptest - Attempted',
              'DoS Slowloris - Attempted',
                               'DoS Hulk',
                   'DoS Hulk - Attempted',
                          'DoS GoldenEye',
                             'Heartbleed',
   'Web Attack - Brute Force - Attempted',
               'Web Attack - Brute Force',
                           'Infiltration',
                'Infiltration - Portscan',
           'Web Attack - XSS - Attempted',
                       'Web Attack - XSS',
 'Web Attack - SQL Injection - Attempted',
               'Infiltration - Attempted',
             'Web Attack - SQL Injection',
                     'Botnet - Attempted',
                                 'B

In [6]:
df.columns

Index(['id', 'FlowID', 'SrcIP', 'SrcPort', 'DstIP', 'DstPort', 'Protocol',
       'Timestamp', 'FlowDuration', 'TotalFwdPacket',
       ...
       'BwdSegmentPayloadLengthMax', 'BwdSegmentPayloadLengthMin',
       'BwdSegmentPayloadLengthMean', 'BwdSegmentPayloadLengthStd',
       'SegmentPayloadLengthMax', 'SegmentPayloadLengthMin',
       'SegmentPayloadLengthMean', 'SegmentPayloadLengthStd', 'Label',
       'AttemptedCategory'],
      dtype='str', length=106)

In [7]:
print(df.isna().any().any())
print(df.columns[df.isna().any()])

False
Index([], dtype='str')


In [8]:
srcPort_name = 'SourcePort'
dstPort_name = 'DestinationPort'
srcIP_name = 'SourceIP'
dstIP_name = 'DestinationIP'

internal_network1 = "192.168.0.0/16"
internal_network2 = "8.6.0.0/16"


df = uniformPorts(df, srcPort_name, dstPort_name)
df = uniformIP(df, srcIP_name, dstIP_name, internal_network=internal_network1, internal_network2=internal_network2)
df.head()

/tmp/ipykernel_69162/3437851558.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['SrcPort_num'] = pd.to_numeric(df['SrcPort'], errors='coerce')
/tmp/ipykernel_69162/3437851558.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['DstPort_num'] = pd.to_numeric(df['DstPort'], errors='coerce')
/tmp/ipykernel_69162/3437851558.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.

int_int = 1070103
 int_ext = 590586
 ext_int = 443624
 ext_ext = 2


,id,FlowID,SrcIP,SrcPort,DstIP,DstPort,Protocol,Timestamp,FlowDuration,TotalFwdPacket,...,SegmentPayloadLengthMean,SegmentPayloadLengthStd,Label,AttemptedCategory,SrcPort_num,DstPort_num,SrcPort_type,DstPort_type,DstIP_internal,SrcIP_internal
0,1,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,2017-07-03 11:56:34.157427,104960112,187,...,0.000000,0.000000,BENIGN,-1,0,0,well-known,well-known,False,True
1,2,192.168.10.9-192.168.10.3-123-123-17,192.168.10.9,123,192.168.10.3,123,17,2017-07-03 11:56:55.428911,65511209,6,...,48.000000,0.000000,BENIGN,-1,123,123,well-known,well-known,True,True
2,3,192.168.10.12-224.0.0.251-5353-5353-17,192.168.10.12,5353,224.0.0.251,5353,17,2017-07-03 11:57:21.057686,113976922,267,...,76.580524,44.140625,BENIGN,-1,5353,5353,registered,registered,False,True
3,4,192.168.10.12-152.2.133.52-123-123-17,192.168.10.12,123,152.2.133.52,123,17,2017-07-03 11:57:31.568196,67037196,8,...,48.000000,0.000000,BENIGN,-1,123,123,well-known,well-known,False,True
4,5,192.168.10.12-45.79.11.217-123-123-17,192.168.10.12,123,45.79.11.217,123,17,2017-07-03 11:57:30.571719,68045057,8,...,48.000000,0.000000,BENIGN,-1,123,123,well-known,well-known,False,True


In [9]:
df.columns

Index(['id', 'FlowID', 'SrcIP', 'SrcPort', 'DstIP', 'DstPort', 'Protocol',
       'Timestamp', 'FlowDuration', 'TotalFwdPacket',
       ...
       'SegmentPayloadLengthMean', 'SegmentPayloadLengthStd', 'Label',
       'AttemptedCategory', 'SrcPort_num', 'DstPort_num', 'SrcPort_type',
       'DstPort_type', 'DstIP_internal', 'SrcIP_internal'],
      dtype='str', length=112)

In [10]:
#NOTE: Change w.r.t. the original pragmatic assessment: Attempted are not labeled as BENIGN
#df['Label2'] = np.where(df['Label'].str.contains('Attempted'), 'BENIGN', df['Label'])
df['Label2'] = df['Label']
df['Label2'] = np.where(df['Label2'].str.contains('Botnet'), 'Botnet', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('SSH-Patator'), 'SSH-Patator', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('FTP-Patator'), 'FTP-Patator', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('DoS slowloris'), 'DoS-Slowloris', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('DoS Slowloris'), 'DoS-Slowloris', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('DoS Slowhttptest'), 'DoS-Slowhttptest', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('DoS Hulk'), 'DoS-Hulk', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('DoS GoldenEye'), 'DoS-Goldeneye', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('DDoS'), 'DoS-Ddos', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('Web Attack - Brute Force'), 'WebAttack-Bruteforce', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('Web Attack - Sql Injection'), 'WebAttack-Sqlinjection', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('Web Attack - SQL Injection'), 'WebAttack-Sqlinjection', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('Web Attack - XSS'), 'WebAttack-Xss', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('Infiltration'), 'Infiltration', df['Label2'])
df['Label2'] = np.where(df['Label2'].str.contains('Portscan'), 'PortScan', df['Label2'])
df['Label_original'] = df['Label']
df['Label'] = df['Label2']
df = df.drop(['Label2'], axis=1)


/tmp/ipykernel_69162/712505242.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Label2'] = df['Label']
/tmp/ipykernel_69162/712505242.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Label_original'] = df['Label']


In [11]:
print(df['Label_original'].unique())

<ArrowStringArray>
[                                'BENIGN',
                'FTP-Patator - Attempted',
                            'FTP-Patator',
                            'SSH-Patator',
                          'DoS Slowloris',
                       'DoS Slowhttptest',
           'DoS Slowhttptest - Attempted',
              'DoS Slowloris - Attempted',
                               'DoS Hulk',
                   'DoS Hulk - Attempted',
                          'DoS GoldenEye',
                             'Heartbleed',
   'Web Attack - Brute Force - Attempted',
               'Web Attack - Brute Force',
                           'Infiltration',
                'Infiltration - Portscan',
           'Web Attack - XSS - Attempted',
                       'Web Attack - XSS',
 'Web Attack - SQL Injection - Attempted',
               'Infiltration - Attempted',
             'Web Attack - SQL Injection',
                     'Botnet - Attempted',
                                 'B

In [12]:
print(df['Label'].unique())

<ArrowStringArray>
[                'BENIGN',            'FTP-Patator',            'SSH-Patator',
          'DoS-Slowloris',       'DoS-Slowhttptest',               'DoS-Hulk',
          'DoS-Goldeneye',             'Heartbleed',   'WebAttack-Bruteforce',
           'Infiltration',          'WebAttack-Xss', 'WebAttack-Sqlinjection',
                 'Botnet',               'PortScan',               'DoS-Ddos']
Length: 15, dtype: str


In [13]:
df.columns

Index(['id', 'FlowID', 'SrcIP', 'SrcPort', 'DstIP', 'DstPort', 'Protocol',
       'Timestamp', 'FlowDuration', 'TotalFwdPacket',
       ...
       'SegmentPayloadLengthStd', 'Label', 'AttemptedCategory', 'SrcPort_num',
       'DstPort_num', 'SrcPort_type', 'DstPort_type', 'DstIP_internal',
       'SrcIP_internal', 'Label_original'],
      dtype='str', length=113)

In [14]:
summa = 0
print("Overall samples: ", len(df))
benign_df = df[df['Label']=='BENIGN']
summa = summa + len(benign_df)
print("\t Benign: ", len(benign_df))

bot_df = df[df['Label']=='Botnet']
summa = summa + len(bot_df)
print("\t Botnet: ", len(bot_df))


portscan_df = df[df['Label']=='PortScan']
summa = summa + len(portscan_df)
print("\t PortScan: ", len(portscan_df))

ddos_df = df[df['Label']=='DoS-Ddos']
summa = summa + len(ddos_df)
print("\t DDoS: ", len(ddos_df))

ftp_df = df[df['Label']=='FTP-Patator']
summa = summa + len(ftp_df)
print("\t FTP-Patator: ", len(ftp_df))


ssh_df = df[df['Label']=='SSH-Patator']
summa = summa + len(ssh_df)
print("\t SSH-Patator: ", len(ssh_df))


slowloris_df = df[df['Label']=='DoS-Slowloris']
summa = summa + len(slowloris_df)
print("\t DoS-Slowloris: ", len(slowloris_df))

slowhttp_df = df[df['Label']=='DoS-Slowhttptest']
summa = summa + len(slowhttp_df)
print("\t DoS-Slowhttp: ", len(slowhttp_df))

hulk_df = df[df['Label']=='DoS-Hulk']
summa = summa + len(hulk_df)
print("\t DoS-Hulk: ", len(hulk_df))

goldeneye_df = df[df['Label']=='DoS-Goldeneye']
summa = summa + len(goldeneye_df)
print("\t DoS-Goldeneye: ", len(goldeneye_df))

hb_df = df[df['Label']=='Heartbleed']
summa = summa + len(hb_df)
print("\t Heartbleed: ", len(hb_df))

brute_df = df[df['Label']=='WebAttack-Bruteforce']
summa = summa + len(brute_df)
print("\t WebAttack-Bruteforce: ", len(brute_df))

inf_df = df[df['Label']=='Infiltration']
summa = summa + len(inf_df)
print("\t Infiltration: ", len(inf_df))

xss_df = df[df['Label']=='WebAttack-Xss']
summa = summa + len(xss_df)
print("\t WebAttack-Xss: ", len(xss_df))

sql_df = df[df['Label']=='WebAttack-Sqlinjection']
summa = summa + len(sql_df)
print("\t WebAttack-Sqlinjection: ", len(sql_df))

print(summa)

Overall samples:  2104315
	 Benign:  1585356
	 Botnet:  4803
	 PortScan:  159656
	 DDoS:  95683
	 FTP-Patator:  4003
	 SSH-Patator:  2988
	 DoS-Slowloris:  5731
	 DoS-Slowhttp:  5111
	 DoS-Hulk:  159140
	 DoS-Goldeneye:  7780
	 Heartbleed:  11
	 WebAttack-Bruteforce:  151
	 Infiltration:  73211
	 WebAttack-Xss:  673
	 WebAttack-Sqlinjection:  18
2104315


In [15]:
other_df = pd.concat([bot_df, hb_df, inf_df, xss_df, sql_df, brute_df])
len(other_df)

78867

In [16]:
malicious_output_folder = os.path.join(output_folder, "malicious")

if not os.path.exists(malicious_output_folder):
    os.makedirs(malicious_output_folder)

other_df.to_csv(os.path.join(malicious_output_folder, "other.csv"))

small_malicious_output_folder = os.path.join(malicious_output_folder, "small")

if not os.path.exists(small_malicious_output_folder):
    os.makedirs(small_malicious_output_folder)

benign_file = os.path.join(output_folder, "benign.csv")

benign_df.to_csv(benign_file)

bot_df.to_csv(os.path.join(small_malicious_output_folder, "bot.csv"))
hb_df.to_csv(os.path.join(small_malicious_output_folder, "heartbleed.csv"))
brute_df.to_csv(os.path.join(small_malicious_output_folder, "webattack-brute.csv"))
inf_df.to_csv(os.path.join(small_malicious_output_folder, "infiltration.csv"))
xss_df.to_csv(os.path.join(small_malicious_output_folder, "webattack-xss.csv"))
sql_df.to_csv(os.path.join(small_malicious_output_folder, "webattack-sql.csv"))

portscan_df.to_csv(os.path.join(malicious_output_folder, "portscan.csv"))
ddos_df.to_csv(os.path.join(malicious_output_folder, "dos-ddos.csv"))
ftp_df.to_csv(os.path.join(malicious_output_folder, "ftp-patator.csv"))
ssh_df.to_csv(os.path.join(malicious_output_folder, "ssh-patator.csv"))
slowloris_df.to_csv(os.path.join(malicious_output_folder, "dos-slowloris.csv"))
slowhttp_df.to_csv(os.path.join(malicious_output_folder, "dos-slowhttp.csv"))
hulk_df.to_csv(os.path.join(malicious_output_folder, "dos-hulk.csv"))
goldeneye_df.to_csv(os.path.join(malicious_output_folder, "dos-goldeneye.csv"))